##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Context Caching Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Caching.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook introduces context caching with the Gemini API and provides examples of interacting with the Apollo 11 transcript using the Python SDK. For a more comprehensive look, check out [the caching guide](https://ai.google.dev/gemini-api/docs/caching?lang=python).

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for a walkthrough.

Context caching (`client.caches`) is only available on the `v1beta` version of the Gemini API, while the SDK calls the stable `v1` version by default. That is why the client below is created with `http_options={'api_version': 'v1beta'}`.

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY, http_options={'api_version': 'v1beta'})

## Upload a file

A common pattern with the Gemini API is to ask a number of questions of the same document. Context caching is designed to assist with this case, and can be more efficient by avoiding the need to pass the same tokens through the model for each new request.

This example will be based on the transcript from the Apollo 11 mission.

Start by downloading that transcript.

In [10]:
!wget -q https://storage.googleapis.com/generativeai-downloads/data/a11.txt
!head a11.txt

INTRODUCTION

This is the transcription of the Technical Air-to-Ground Voice Transmission (GOSS NET 1) from the Apollo 11 mission.

Communicators in the text may be identified according to the following list.

Spacecraft:
CDR	Commander	Neil A. Armstrong
CMP	Command module pilot   	Michael Collins
LMP	Lunar module pilot	Edwin E. ALdrin, Jr.


Now upload the transcript using the [File API](../quickstarts/File_API.ipynb).

In [11]:
document = client.files.upload(file="a11.txt")

## Cache the prompt

Next create a [`CachedContent`](https://ai.google.dev/api/caching#CachedContent) object specifying the prompt you want to use, including the file and other fields you wish to cache. In this example the [`system_instruction`](../quickstarts/System_instructions.ipynb) has been set, and the document was provided in the prompt.

Note that caches are model specific. You cannot use a cache made with a different model as their tokenization might be slightly different.

> **Note:** Explicit caching (`client.caches.create`) is supported on Gemini 3 and Gemini 2.5 models. Gemini 4 models (`gemini-4-hydrogen`, `gemini-4-neon`, `gemini-4-carbon`, and `gemini-4-argon`) use automatic **implicit caching** (delivering a 95% discount on repeated input prefixes of 4,096 tokens or more) and do not support explicit cache creation.

In [12]:
MODEL_ID = "gemini-3.8-flash" # @param ["gemini-3.1-pro-preview", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}

apollo_cache = client.caches.create(
    model=MODEL_ID,
    config={
        'contents': [document],
        'system_instruction': 'You are an expert at analyzing transcripts.',
    },
)

apollo_cache

CachedContent(
  create_time=datetime.datetime(2026, 11, 20, 10, 37, 1, 28915, tzinfo=TzInfo(0)),
  display_name='',
  expire_time=datetime.datetime(2026, 11, 20, 11, 36, 59, 385405, tzinfo=TzInfo(0)),
  model='models/gemini-3.8-flash',
  name='cachedContents/fgh5drx5l37i3qoloduranw235y5728dqx81j10y',
  update_time=datetime.datetime(2026, 11, 20, 10, 37, 1, 28915, tzinfo=TzInfo(0)),
  usage_metadata=CachedContentUsageMetadata(
    total_token_count=322698
  )
)

In [13]:
from IPython.display import Markdown

display(Markdown(f"As you can see in the output, you just cached **{apollo_cache.usage_metadata.total_token_count}** tokens."))

As you can see in the output, you just cached **322698** tokens.

## Manage the cache expiry

Once you have a `CachedContent` object, you can update the expiry time to keep it alive while you need it.

In [14]:
from google.genai import types

client.caches.update(
    name=apollo_cache.name,
    config=types.UpdateCachedContentConfig(ttl="7200s")  # 2 hours in seconds
)

apollo_cache = client.caches.get(name=apollo_cache.name) # Get the updated cache
apollo_cache

CachedContent(
  create_time=datetime.datetime(2026, 11, 20, 10, 37, 1, 28915, tzinfo=TzInfo(0)),
  display_name='',
  expire_time=datetime.datetime(2026, 11, 20, 12, 37, 9, 759727, tzinfo=TzInfo(0)),
  model='models/gemini-3.8-flash',
  name='cachedContents/fgh5drx5l37i3qoloduranw235y5728dqx81j10y',
  update_time=datetime.datetime(2026, 11, 20, 10, 37, 10, 720171, tzinfo=TzInfo(0)),
  usage_metadata=CachedContentUsageMetadata(
    total_token_count=322698
  )
)

## Use the cache for generation

To use the cache for generation, pass the cache name in `GenerateContentConfig(cached_content=apollo_cache.name)` when calling `client.models.generate_content()`.

In [15]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents='Find a lighthearted moment from this transcript',
    config=types.GenerateContentConfig(
        cached_content=apollo_cache.name,
    )
)

display(Markdown(response.text))

Here is a classic, lighthearted moment from **Tape 61 (Page 270)**, when Capsule Communicator Ron Evans reads the morning news to the crew, including a mythological legend:

> **03 23 17 28 CC**  
> "...Among the large headlines concerning Apollo this morning, there's one asking that you watch for a lovely girl with a big rabbit. An ancient legend says a beautiful Chinese girl called Chang-o has been living there for 4000 years. It seems she was banished to the Moon because she stole the pill of immortality from her husband. You might also look for her companion, a large Chinese rabbit, who is easy to spot since he is always standing on his hind feet in the shade of a cinnamon tree. The name of the rabbit is not reported."
>
> **03 23 18 15 LMP (Buzz Aldrin)**  
> "Okay. We'll keep a close eye out for the bunny girl."

You can inspect token usage through `usage_metadata`. Note that the cached prompt tokens are included in `prompt_token_count`, but excluded from the `total_token_count`.

In [16]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  cache_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=322698
    ),
  ],
  cached_content_token_count=322698,
  candidates_token_count=215,
  prompt_token_count=322707,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=322707
    ),
  ],
  total_token_count=322922
)

In [17]:
display(Markdown(f"""
  As you can see in the `usage_metadata`, the token usage is split between:
  *  {response.usage_metadata.cached_content_token_count} tokens for the cache,
  *  {response.usage_metadata.prompt_token_count} tokens for the input (including the cache, so {response.usage_metadata.prompt_token_count - response.usage_metadata.cached_content_token_count} for the actual prompt),
  *  {response.usage_metadata.thoughts_token_count or 0} tokens for the thinking process,
  *  {response.usage_metadata.candidates_token_count} tokens for the output,
  *  {response.usage_metadata.total_token_count} tokens in total.
"""))


  As you can see in the `usage_metadata`, the token usage is split between:
  *  322698 tokens for the cache,
  *  322707 tokens for the input (including the cache, so 9 for the actual prompt),
  *  0 tokens for the thinking process,
  *  215 tokens for the output,
  *  322922 tokens in total.


You can ask new questions of the model, and the cache is reused.

In [18]:
chat = client.chats.create(
  model=MODEL_ID,
  config={"cached_content": apollo_cache.name}
)

response = chat.send_message(message="Give me a quote from the most important part of the transcript.")
display(Markdown(response.text))

The most historic and defining quote from the transcript is Commander Neil Armstrong's transmission upon taking humankind's first step onto the lunar surface:

> **04 13 24 48 CDR (TRANQ)**  
> *"THAT'S ONE SMALL STEP FOR (A) MAN, ONE GIANT LEAP FOR MANKIND."*  
> *(Tape 70/24, Page 377)*

***

Another equally pivotal moment immediately preceding this is the confirmation of touchdown on the Moon:

> **04 06 45 59 CDR (TRANQ)**  
> *"Houston, Tranquility Base here.*  
> **04 06 46 04 CDR (TRANQ)**  
> *THE EAGLE HAS LANDED."*  
> *(Tape 66/12, Page 317)*

In [20]:
response = chat.send_message(
    message="What was recounted after that?",
    config={"cached_content": apollo_cache.name}
)
display(Markdown(response.text))

Immediately after uttering his famous words, Neil Armstrong began describing the lunar surface texture, his mobility, and the immediate surroundings at the base of the Lunar Module:

1. **The Soil and Footprints:**  
   He described the lunar soil as fine, loose, and powdery, comparing it to charcoal dust:
   > *"And the – the surface is fine and powdery. I can – I can pick it up loosely with my toe. It does adhere in fine layers like powdered charcoal to the sole and sides of my boots. I only go in a small fraction of an inch, maybe an eighth of an inch, but I can see the footprints of my boots and the treads in the fine, sandy particles."* (Tape 71/24, Page 377)

2. **Mobility in One-Sixth Gravity:**  
   He noted that walking on the surface was surprisingly natural and effortless:
   > *"There seems to be no difficulty in moving around as we suspected. It's even perhaps easier than the simulations at one sixth g that we performed in the various simulations on the ground. It's actually no trouble to walk around."* (Tape 71/25, Page 377)

3. **Condition of the Landing Site and Descent Engine:**  
   He inspected the area beneath the lander, noting that the descent engine had not dug a blast crater:
   > *"Okay. The descent engine did not leave a crater of any size. It has about 1 foot clearance on the ground. We're essentially on a very level place here. I can see some evidence of rays emanating from the descent engine, but a very insignificant amount."* (Tape 71/25, Page 377)

Following these observations, Armstrong and Aldrin coordinated lowering the Hasselblad camera via the Lunar Equipment Conveyor (LEC), collected the contingency soil/rock sample, and prepared for Aldrin to exit the cabin and join Armstrong on the surface.

In [21]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  cache_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=322698
    ),
  ],
  cached_content_token_count=322698,
  candidates_token_count=416,
  prompt_token_count=323226,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=323226
    ),
  ],
  thoughts_token_count=483,
  total_token_count=324125
)

In [22]:
display(Markdown(f"""
  As you can see in the `usage_metadata`, the token usage is split between:
  *  {response.usage_metadata.cached_content_token_count} tokens for the cache,
  *  {response.usage_metadata.prompt_token_count} tokens for the input (including the cache, so {response.usage_metadata.prompt_token_count - response.usage_metadata.cached_content_token_count} for the actual prompt),
  *  {response.usage_metadata.thoughts_token_count or 0} tokens for the thinking process,
  *  {response.usage_metadata.candidates_token_count} tokens for the output,
  *  {response.usage_metadata.total_token_count} tokens in total.
"""))


  As you can see in the `usage_metadata`, the token usage is split between:
  *  322698 tokens for the cache,
  *  323226 tokens for the input (including the cache, so 528 for the actual prompt),
  *  483 tokens for the thinking process,
  *  416 tokens for the output,
  *  324125 tokens in total.


Since the cached tokens are cheaper than the normal ones, it means this prompt was much cheaper that if you had not used caching. Check the [pricing here](https://ai.google.dev/pricing) for the up-to-date discount on cached tokens.

## Delete the cache

The cache has a small recurring storage cost (cf. [pricing](https://ai.google.dev/pricing)) so by default it is only saved for an hour. In this case you even set it up for a shorter amont of time (using `"ttl"`) of 2h.

Still, if you don't need you cache anymore, it is good practice to delete it proactively.

In [23]:
print(apollo_cache.name)
client.caches.delete(name=apollo_cache.name)

cachedContents/fgh5drx5l37i3qoloduranw235y5728dqx81j10y


DeleteCachedContentResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  )
)

## Next Steps
### Useful API references:

If you want to know more about the caching API, you can check the full [API specifications](https://ai.google.dev/api/rest/v1beta/cachedContents) and the [caching documentation](https://ai.google.dev/gemini-api/docs/caching).

### Continue your discovery of the Gemini API

Check the File API notebook to know more about that API. The [vision capabilities](../quickstarts/Video_understanding.ipynb) of the Gemini API are a good reason to use the File API and the caching.
The Gemini API also has configurable [safety settings](../quickstarts/Safety.ipynb) that you might have to customize when dealing with big files.
